In [2]:
# Cell 1: Imports and Setup
import mne
import os
import numpy as np
import matplotlib.pyplot as plt
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf
import  colorama

# Ensure matplotlib plots display inline in the notebook
%matplotlib inline

# Set MNE logging level to 'WARNING' to reduce text output clutter in the notebook
mne.set_log_level('WARNING')

# Define exclusions based on known dataset issues
# Subject 88: Sampled at 128 Hz instead of 160 Hz
# Subjects 38, 89, 92, 100, 104, 106: Known annotation/event errors
# EXCLUDED_SUBJECTS = [38, 88, 89, 92, 100, 104, 106]
EXCLUDED_SUBJECTS = []


# Downloading and Parsing the Data

In [ ]:
def load_eeg_data(subject_id, run_id, base_path="./data/files"):
    """
    Loads an EDF+ file for a specific subject and run, verifying metadata constraints.
    """
    if subject_id in EXCLUDED_SUBJECTS:
        raise ValueError(f"Subj
ect {subject_id} is excluded from this analysis due to dataset anomalies.")
        
    # Format strings to match PhysioNet standard (e.g., S001/S001R04.edf)
    subj_str = f"S{subject_id:03d}"
    run_str = f"R{run_id:02d}"
    file_path = os.path.join(base_path, subj_str, f"{subj_str}{run_str}.edf")
    
    # Check if the file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Data file not found at: {file_path}")
        
    # Load the EDF file using MNE this should respect path standards (e.g., S001/S001R04.edf)
    print(f"Loading data from {file_path}...")
    raw = mne.io.read_raw_edf(file_path, preload=True)
    
    # 1. Verify Sampling Rate
    if raw.info['sfreq'] != 160.0:
        raise ValueError(f"Sampling rate mismatch! Expected 160 Hz, got {raw.info['sfreq']} Hz")
        
    # 2. Verify Channel Count
    if len(raw.ch_names) != 64:
        raise ValueError(f"Channel count mismatch! Expected 64, got {len(raw.ch_names)}")
        
    # 3. Standardize Channel Names & Set Montage
    # PhysioNet EDF files often have trailing dots in channel names (e.g., 'Fc5.')
    mne.datasets.eegbci.standardize(raw) 
    # Set the standard 10-20 montage for EEG channel locations
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage)
    
    print("Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.")
    return raw

In [17]:
# Cell 2: Downloading and Parsing the Data

# Target all 109 SUBJECT_TO_TEST available in the dataset
# Note: To test quickly, change this to range(1, 3)
SUBJECT_TO_TEST = list(range(1, 110))
# SUBJECT_TO_TEST = list(range(1, 2))


In [18]:
print(f"Subjects to test: {SUBJECT_TO_TEST}")

Subjects to test: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109]


In [4]:

#! Define the runs you want to analyze.
# Ligne de base (Baseline), yeux ouverts
run_open_eyes = [1]
# Ligne de base (Baseline), yeux fermés
run_closed_eyes = [2]
# Motor execution: Open and close fist (Left vs. Right)
run_execution_hand = [3, 7, 11]
# Motor imagery: Imagine opening and closing fist (Left vs. Right)
run_imagery_hand = [4, 8, 12]
# Motor execution: Open and close both fists vs. both feet
run_execution_both_hands_feet = [5, 9, 13]
# Motor imagery: Imagine opening and closing both fists vs. both feet
run_imagery_both_hands_feet = [6, 10, 14]

# runs the entire set of runs for the BCI Competition IV dataset, which includes baseline, motor execution, and motor imagery tasks
runs = (
    run_open_eyes
    + run_closed_eyes
    + run_execution_hand
    + run_imagery_hand
    + run_execution_both_hands_feet
    + run_imagery_both_hands_feet
)

In [12]:
runs

[1, 2, 3, 7, 11, 4, 8, 12, 5, 9, 13, 6, 10, 14]

In [13]:
len(runs)

14

In [14]:

# Define your local data path (this should match your .gitignore)
data_path = "./data/files"

In [10]:
raw = load_eeg_data(subject_id=1, run_id=1, base_path=data_path)

Loading data from ./data/files/S001/S001R01.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.


In [21]:

print(
    f"Initiating download for {len(SUBJECT_TO_TEST)} SUBJECT_TO_TEST. This may take a while..."
)

# Initialize an empty list to hold our raw data objects
all_raws = []

for subject in SUBJECT_TO_TEST:
    for run in runs:
        try:
            # Concatenate the runs for this subject into a single continuous object
            subject_raw = load_eeg_data(subject, run, base_path=data_path)

            # Append to our master list
            all_raws.append(subject_raw)

        except Exception as e:
            print(f"Could not load data for subject {subject}: {e}")

# If you want to merge all SUBJECT_TO_TEST into one massive continuous recording (use with caution regarding RAM):
# full_dataset = concatenate_raws(all_raws)

print(f"Successfully loaded data for {len(all_raws)} SUBJECT_TO_TEST.")

Initiating download for 109 SUBJECT_TO_TEST. This may take a while...
Loading data from ./data/files/S001/S001R01.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S001/S001R02.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S001/S001R03.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S001/S001R07.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S001/S001R11.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S001/S001R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S001/S001R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_

/var/folders/lt/0wgz0hyd67z6zxh62hn6qff40000gn/T/ipykernel_3349/1891292187.py:19: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True)
/var/folders/lt/0wgz0hyd67z6zxh62hn6qff40000gn/T/ipykernel_3349/1891292187.py:19: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True)
/var/folders/lt/0wgz0hyd67z6zxh62hn6qff40000gn/T/ipykernel_3349/1891292187.py:19: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True)
/var/folders/lt/0wgz0hyd67z6zxh62hn6qff40000gn/T/ipykernel_3349/1891292187.py:19: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True)
/var/folders/lt/0wgz0hyd67z6zxh62hn6qff40000gn/T/ipykernel_3349/1891292187.py:19: RuntimeWarning: Limited 1 annotation(s) th

Could not load data for subject 100: Sampling rate mismatch! Expected 160 Hz, got 128.0 Hz
Loading data from ./data/files/S100/S100R04.edf...
Could not load data for subject 100: Sampling rate mismatch! Expected 160 Hz, got 128.0 Hz
Loading data from ./data/files/S100/S100R08.edf...
Could not load data for subject 100: Sampling rate mismatch! Expected 160 Hz, got 128.0 Hz
Loading data from ./data/files/S100/S100R12.edf...
Could not load data for subject 100: Sampling rate mismatch! Expected 160 Hz, got 128.0 Hz
Loading data from ./data/files/S100/S100R05.edf...
Could not load data for subject 100: Sampling rate mismatch! Expected 160 Hz, got 128.0 Hz
Loading data from ./data/files/S100/S100R09.edf...
Could not load data for subject 100: Sampling rate mismatch! Expected 160 Hz, got 128.0 Hz
Loading data from ./data/files/S100/S100R13.edf...
Could not load data for subject 100: Sampling rate mismatch! Expected 160 Hz, got 128.0 Hz
Loading data from ./data/files/S100/S100R06.edf...
Could 

/var/folders/lt/0wgz0hyd67z6zxh62hn6qff40000gn/T/ipykernel_3349/1891292187.py:19: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S101/S101R02.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S101/S101R03.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S101/S101R07.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S101/S101R11.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S101/S101R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S101/S101R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from ./data/files/S101/S101R12.edf...
Metadata verified: 160 Hz sampling rate, 64 chan

# Cell 3: Data Parsing, Validation & Event Extraction


In [10]:
all_raws[0]

<RawEDF | S001R04.edf, 64 x 60000 (375.0 s), ~29.3 MiB, data loaded>

In [ ]:
len(all_raws) #  

2

In [ ]:


# For demonstration, we will process the first subject's raw data from our 'all_raws' list
raw = all_raws[0]


In [ ]:
all_raws[0].info.keys()

dict_keys(['acq_pars', 'acq_stim', 'ctf_head_t', 'description', 'dev_head_t', 'dev_ctf_t', 'dig', 'experimenter', 'utc_offset', 'device_info', 'file_id', 'highpass', 'hpi_subsystem', 'kit_system_id', 'helium_info', 'line_freq', 'lowpass', 'meas_date', 'meas_id', 'proj_id', 'proj_name', 'subject_info', 'xplotter_layout', 'gantry_angle', 'bads', 'chs', 'comps', 'events', 'hpi_meas', 'hpi_results', 'projs', 'proc_history', 'custom_ref_applied', 'sfreq', 'ch_names', 'nchan'])

In [36]:
# ==========================================
# !1. Validate Channel Count & Sampling Rate
# ==========================================
print(
    colorama.Style.BRIGHT
    + colorama.Fore.GREEN
    + "--- Data Validation ---"
    + colorama.Style.RESET_ALL
)

for index, subject_raw in enumerate(all_raws):

    # nchan is the number of channels in the EEG (Electroencephalography) data
    n_channels = subject_raw.info["nchan"]
    # sfreq is the sampling frequency of the EEG data, which indicates how many samples per second were recorded
    sfreq = subject_raw.info["sfreq"]

    # Check for the expected 160 Hz rate and 64 channels
    # the frequency of 160 hz is a common sampling rate for EEG data
    if sfreq != 160.0:
        print(f"{colorama.Fore.RED}Subject {index + 1}:")
        print(f"Sampling Frequency: {sfreq} Hz")
        print(
            "⚠️ WARNING: Sampling rate is not 160 Hz. Consider excluding or resampling this subject."
        )

    # channel should be 64 for the BCI Competition IV dataset, which is a common standard for EEG datasets
    # one channel per electrode is used to capture the electrical activity of the brain, and 64 covers a wide area of the scalp,
    # providing a good balance between spatial resolution and computational efficiency
    if n_channels != 64:
        print(f"{colorama.Fore.RED}Subject {index + 1}:")
        print(f"Number of Channels: {n_channels}")
        print("⚠️ WARNING: Channel count is not 64. Check the dataset integrity.")

print(
    f"{colorama.Fore.GREEN}✅ for all the rest subjects, the sampling rate is exactly 160 Hz and the channel count is 64, which are the expected values for this dataset."
)

--- Data Validation ---
Subject 88:
Sampling Frequency: 128.0 Hz
⚠️ WARNING: Sampling rate is not 160 Hz. Consider excluding or resampling this subject.
Subject 92:
Sampling Frequency: 128.0 Hz
⚠️ WARNING: Sampling rate is not 160 Hz. Consider excluding or resampling this subject.
Subject 100:
Sampling Frequency: 128.0 Hz
⚠️ WARNING: Sampling rate is not 160 Hz. Consider excluding or resampling this subject.
✅ for all the rest subjects, the sampling rate is exactly 160 Hz and the channel count is 64, which are the expected values for this dataset.


In [1]:

# ==========================================
# 2. Extract and Map Event Codes
# ==========================================
# The EDF+ format stores events as string annotations ('T0', 'T1', 'T2').
# We must convert these strings into a numeric matrix (events array) for scikit-learn.

# Define the custom mapping explicitly based on your requirements
custom_mapping = {
    'T0': 0,  # Rest
    'T1': 1,  # Left fist motion/imagery
    'T2': 2   # Right fist motion/imagery
}

# Extract the events using the custom mapping
events, event_dict = mne.events_from_annotations(raw, event_id=custom_mapping)

print("\n--- Event Extraction ---")
print(f"Total events found: {len(events)}")
print("Event Dictionary Mapping:", event_dict)

# Quick sanity check to see the first 5 events (Timestamp, duration, event_code)
print("\nFirst 5 event triggers (Sample Index, Previous Event, Event Code):")
print(events[:5])

NameError: name 'mne' is not defined